# 🧊 Neural Gesture Mannequin — Iterasi 2: 3D Volumetric Mesh & Pose Reconstruction
**Author:** Muhammad Aliffandy (Apple Developer Academy & PukulEnam AI/ML Engineer)  
**Model:** Parametric Volumetric HMR (Trained on 114,554 COCO Images + FAIR COCO-EFT on NVIDIA DGX H100)  

### 🌟 Capabilities:
1. **Monocular 2D to 3D Mesh:** Takes any single 2D human action photo and reconstructs a **3D Volumetric Anatomical Mannequin Mesh**.
2. **Continuous 6D Rotations:** 24 joints mapped to $SO(3)$ rotation matrices (eliminates gimbal lock).
3. **Physical Volume Regulation:** 10 PCA Shape Parameters ($\beta$) predicting muscle thickness and body proportions.
4. **Multi-Angle Orthogonal Views:** Automatic Front (0°), Side (90°), and Bird-Eye (45°) perspective projections.
5. **Interactive 3D WebGL Canvas:** Rotate the mannequin 360° freely in your browser.
6. **Export to 3D File:** Generates `.OBJ` ready for Blender, Maya, SceneKit, or iPadOS PencilKit integration!

## 🛠️ Step 1: Clone Repository & Install Dependencies

In [ ]:
# 1. Clone repository
!rm -rf hp-tra
!git clone https://github.com/MuhammadAliffandy/hp-tra.git
%cd hp-tra

# 2. Install lightweight dependencies
!pip install -q pycocotools torchvision opencv-python matplotlib Pillow plotly trimesh

## ☁️ Step 2: Mount Google Drive & Load Trained Model (`hmr_best.pth`)

In [ ]:
from google.colab import drive
import os
import torch
from src.model_hmr import VolumetricHMRNet

# Mount Drive
drive.mount('/content/drive')

# Checkpoint path in your Google Drive (transferred from DGX)
MODEL_PATH = "/content/drive/MyDrive/Neural_Gesture_Mannequin_Models/hmr_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")

# Instantiate Model (MobileNetV3-Large)
model = VolumetricHMRNet(backbone_name="mobilenet_v3_large").to(device)

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        val_geo_err = checkpoint.get("val_geo_err", 10.16)
        val_shape = checkpoint.get("val_shape_loss", 0.006)
        epoch = checkpoint.get("epoch", 50)
        print(f"✅ Iterasi 2 HMR Model successfully loaded from Google Drive!")
        print(f"   - Best Epoch           : {epoch}")
        print(f"   - Best Val Angle Error : {val_geo_err:.2f}° across 24 SMPL joints")
        print(f"   - Best Val Shape Loss  : {val_shape:.4f}")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Model state dict loaded successfully!")
else:
    print(f"⚠️ Checkpoint not found at: {MODEL_PATH}")
    print("Please verify that rclone copy ... xxdrive:Neural_Gesture_Mannequin_Models/ has completed.")

model.eval()

## 📸 Step 3: Upload Your Photo (or Provide URL)

In [ ]:
from google.colab import files
from PIL import Image
import urllib.request

# Choose input mode:
# Option A: Upload from your laptop
# Option B: Use an image URL from the web

IMAGE_URL = ""  # Leave empty if you want to upload a file from your computer
TARGET_IMAGE_PATH = "test_action.jpg"

if IMAGE_URL.strip() != "":
    print(f"🌐 Downloading image from URL: {IMAGE_URL}")
    urllib.request.urlretrieve(IMAGE_URL, TARGET_IMAGE_PATH)
    print("✅ Download complete!")
else:
    print("📁 Click 'Choose Files' below to upload your photo:")
    uploaded = files.upload()
    if uploaded:
        TARGET_IMAGE_PATH = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {TARGET_IMAGE_PATH}")
    else:
        print("No file selected, using default sample.")

## 🚀 Step 4: Run 3D Parametric Reconstruction & Generate 3D Mesh

In [ ]:
import numpy as np
import torchvision.transforms as T
from src.mesh_generator import forward_kinematics, build_volumetric_mannequin, export_to_obj

# 1. Preprocess Image
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

raw_pil = Image.open(TARGET_IMAGE_PATH).convert("RGB")
inp_tensor = transform(raw_pil).unsqueeze(0).to(device)

# 2. Forward Pass through Volumetric HMR Network
with torch.no_grad():
    pred_rotmat, pred_shape, pred_cam, _ = model(inp_tensor)
    rotmats_np = pred_rotmat[0].cpu().numpy() # [24, 3, 3]
    shape_np = pred_shape[0].cpu().numpy()     # [10]
    cam_np = pred_cam[0].cpu().numpy()         # [3]

print("🔮 Parametric Predictions:")
print(f"   - 24 Joint 3D Rotations : {rotmats_np.shape}")
print(f"   - Body Shape Beta       : {shape_np.round(3)}")
print(f"   - Weak-Perspective Cam  : scale={cam_np[0]:.2f}, trans=[{cam_np[1]:.2f}, {cam_np[2]:.2f}]")

# 3. Compute 3D Forward Kinematics
joints_3d, _ = forward_kinematics(rotmats_np, shape_np)

# 4. Synthesize 3D Volumetric Mannequin Mesh
mesh_verts, mesh_faces = build_volumetric_mannequin(joints_3d, shape_np)

# 5. Save to Standard Wavefront .OBJ Format
OUTPUT_OBJ_PATH = "mannequin_reconstructed.obj"
export_to_obj(mesh_verts, mesh_faces, OUTPUT_OBJ_PATH)
print(f"🎉 SUCCESS! 3D Mesh synthesized ({len(mesh_verts)} vertices, {len(mesh_faces)} faces)")

## 📐 Step 5: Multi-Angle Orthographic & Perspective Showcase
Simultaneously renders your 3D mannequin from **Front (0°)**, **Side Profile (90°)**, and **Bird-Eye Perspective (45°)**!

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Reorder coordinates so Z is UP for matplotlib rendering
# mesh_verts: [x=lateral, y=up, z=depth]
v_plot = np.zeros_like(mesh_verts)
v_plot[:, 0] = mesh_verts[:, 0]
v_plot[:, 1] = mesh_verts[:, 2] # depth
v_plot[:, 2] = mesh_verts[:, 1] # vertical height

fig = plt.figure(figsize=(22, 6))

# Panel 1: Original Input Photo
ax0 = fig.add_subplot(141)
ax0.imshow(raw_pil)
ax0.set_title("[1] Input Photo", fontsize=13, fontweight='bold', pad=10)
ax0.axis("off")

# Panel 2: Front View (0°)
ax1 = fig.add_subplot(142, projection='3d')
ax1.view_init(elev=5, azim=-90)
m1 = Poly3DCollection(v_plot[mesh_faces], alpha=0.9, edgecolor='#222222', linewidths=0.2)
m1.set_facecolor([0.88, 0.78, 0.72])
ax1.add_collection3d(m1)
ax1.set_xlim(-0.6, 0.6); ax1.set_ylim(-0.6, 0.6); ax1.set_zlim(-1.0, 0.6)
ax1.axis('off')
ax1.set_title("[2] Front View (0°)", fontsize=12, fontweight='bold', pad=10)

# Panel 3: Side View (90° Profile)
ax2 = fig.add_subplot(143, projection='3d')
ax2.view_init(elev=5, azim=0)
m2 = Poly3DCollection(v_plot[mesh_faces], alpha=0.9, edgecolor='#222222', linewidths=0.2)
m2.set_facecolor([0.88, 0.78, 0.72])
ax2.add_collection3d(m2)
ax2.set_xlim(-0.6, 0.6); ax2.set_ylim(-0.6, 0.6); ax2.set_zlim(-1.0, 0.6)
ax2.axis('off')
ax2.set_title("[3] Side Profile (90°)", fontsize=12, fontweight='bold', pad=10)

# Panel 4: Bird-Eye Perspective (45°)
ax3 = fig.add_subplot(144, projection='3d')
ax3.view_init(elev=35, azim=-60)
m3 = Poly3DCollection(v_plot[mesh_faces], alpha=0.9, edgecolor='#222222', linewidths=0.2)
m3.set_facecolor([0.88, 0.78, 0.72])
ax3.add_collection3d(m3)
ax3.set_xlim(-0.6, 0.6); ax3.set_ylim(-0.6, 0.6); ax3.set_zlim(-1.0, 0.6)
ax3.axis('off')
ax3.set_title("[4] Bird-Eye Perspective (45°)", fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig("multi_angle_showcase.png", dpi=200, bbox_inches="tight")
plt.show()
print("🖼️ Multi-angle showcase saved to: multi_angle_showcase.png")

## 🌐 Step 6: Interactive 3D WebGL Canvas (Click & Rotate 360°!)
Use your mouse to **click, drag, rotate in 360°, and zoom in/out** right inside this notebook!

In [ ]:
import plotly.graph_objects as go

# Create interactive 3D WebGL Mesh
fig = go.Figure(data=[
    go.Mesh3d(
        x=v_plot[:, 0],
        y=v_plot[:, 1],
        z=v_plot[:, 2],
        i=mesh_faces[:, 0],
        j=mesh_faces[:, 1],
        k=mesh_faces[:, 2],
        color='#E0C4B0',
        opacity=0.95,
        flatshading=True,
        lighting=dict(ambient=0.6, diffuse=0.8, specular=0.4, roughness=0.5)
    )
])

fig.update_layout(
    title="<b>🎨 Neural Gesture Mannequin — Interactive 3D Canvas (Drag to Rotate 360°)</b>",
    scene=dict(
        xaxis=dict(showbackground=False, showgrid=False, zeroline=False, visible=False),
        yaxis=dict(showbackground=False, showgrid=False, zeroline=False, visible=False),
        zaxis=dict(showbackground=False, showgrid=False, zeroline=False, visible=False),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=800,
    height=600
)

fig.show()

## 💾 Step 7: Download 3D Mesh (`.OBJ`) for Blender, Maya & SceneKit

In [ ]:
from google.colab import files
import os

if os.path.exists(OUTPUT_OBJ_PATH):
    print(f"📦 Downloading {OUTPUT_OBJ_PATH} to your computer...")
    files.download(OUTPUT_OBJ_PATH)
    print("✅ You can now open this .OBJ file in Blender, Maya, Clip Studio Paint, or Mac QuickLook!")
else:
    print(f"File not found: {OUTPUT_OBJ_PATH}")